# BUSI1783 – Board Diversity and Firm Performance

This notebook documents the Python analysis used for the BUSI1783 cross-sectional study.

**Design:** 872 UK publicly listed non-financial firms, one observation per firm at FY0.

**Outcomes:** ROA, ROE and Tobin's Q.

**Predictors:** Board Gender Diversity, Specific Skills, Board Size, Policy Board Experience, Firm Size and Leverage.

**Estimation:** OLS with HC3 heteroskedasticity-robust standard errors and GICS sector fixed effects.

The notebook also produces Pearson correlations, VIF diagnostics, Breusch–Pagan heteroskedasticity tests, Cook's-distance influence checks and sensitivity analyses for ROE and Tobin's Q.


## 1. Setup and data loading

The notebook expects the project dataset at `../data/BUSI1783_Cross_Sectional_Dataset.xlsx` when the notebook is run from the repository's `analysis` folder.

In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")

DATA_FILE = Path("../data/BUSI1783_Cross_Sectional_Dataset.xlsx")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_excel(DATA_FILE)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.head())


In [ ]:

# Source-to-analysis column mapping
COLUMN_MAP = {
    "Board Gender Diversity, Percent (FY0)": "Gender_Diversity",
    "Return On Assets": "ROA",
    "ROE": "ROE",
    "Tobins_Q": "Tobins_Q",
    "Firm Size": "Firm_Size",
    "Board Size (FY0)": "Board_Size",
    "Board Specific Skills, Percent (FY0)": "Specific_Skills",
    "Policy Board Experience (FY0)": "Policy_Experience",
    "Total Debt Percentage of Total Equity (FY0)": "Leverage",
    "GICS Sector Name": "Sector",
    "Identifier (RIC)": "RIC",
    "Company Name": "Company_Name",
}

missing_source = [c for c in COLUMN_MAP if c not in df.columns]
if missing_source:
    raise KeyError(f"Missing expected source columns: {missing_source}")

data = df.rename(columns=COLUMN_MAP).copy()

# Robust parsing for the policy-experience indicator.
def parse_policy_experience(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"true", "yes", "y", "1", "1.0"}:
            return 1.0
        if s in {"false", "no", "n", "0", "0.0"}:
            return 0.0
    try:
        v = float(x)
        if v in (0.0, 1.0):
            return v
    except (TypeError, ValueError):
        pass
    return np.nan

data["Policy_Experience"] = data["Policy_Experience"].map(parse_policy_experience)

numeric_cols = [
    "Gender_Diversity", "ROA", "ROE", "Tobins_Q",
    "Firm_Size", "Board_Size", "Specific_Skills",
    "Policy_Experience", "Leverage"
]
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

print(data[numeric_cols].describe().T)


## 2. Descriptive statistics and missingness

In [ ]:

descriptives = data[numeric_cols].describe().T
missingness = pd.DataFrame({
    "N": data[numeric_cols].notna().sum(),
    "Missing": data[numeric_cols].isna().sum(),
    "Missing_Percent": data[numeric_cols].isna().mean() * 100
})
display(descriptives)
display(missingness)


## 3. Regression models

For each outcome, complete cases are used for the variables required by that model. Sector fixed effects are represented by GICS sector dummy variables, with the first observed sector omitted as the reference category. HC3 robust standard errors are used.

In [ ]:

PREDICTORS = [
    "Gender_Diversity",
    "Specific_Skills",
    "Board_Size",
    "Policy_Experience",
    "Firm_Size",
    "Leverage",
]
OUTCOMES = ["ROA", "ROE", "Tobins_Q"]

def fit_model(outcome):
    cols = [outcome, *PREDICTORS, "Sector"]
    model_data = data[cols].dropna().copy()
    model_data["Sector"] = model_data["Sector"].astype("category")

    X = pd.get_dummies(
        model_data[PREDICTORS + ["Sector"]],
        columns=["Sector"],
        drop_first=True,
        dtype=float,
    )
    X = sm.add_constant(X, has_constant="add")
    y = model_data[outcome].astype(float)

    model = sm.OLS(y, X).fit(cov_type="HC3")
    return model, model_data, X

models = {}
for outcome in OUTCOMES:
    model, model_data, X = fit_model(outcome)
    models[outcome] = (model, model_data, X)
    print(f"\n=== {outcome} ===")
    print("N =", int(model.nobs))
    print("R-squared =", model.rsquared)
    print("Adjusted R-squared =", model.rsquared_adj)
    display(model.summary2().tables[1])


## 4. Pearson correlations

In [ ]:

corr_vars = [
    "ROA", "ROE", "Tobins_Q",
    "Gender_Diversity", "Specific_Skills",
    "Board_Size", "Policy_Experience",
    "Firm_Size", "Leverage"
]
correlations = data[corr_vars].corr(method="pearson")
display(correlations)
correlations.to_csv(OUTPUT_DIR / "correlations.csv")


## 5. Variance Inflation Factors (VIF)

VIF is calculated for the six substantive predictors only. Sector dummy variables are not included in this diagnostic table.

In [ ]:

vif_data = data[PREDICTORS].dropna().astype(float).copy()
vif_X = sm.add_constant(vif_data, has_constant="add")

vif_table = pd.DataFrame({
    "Variable": vif_X.columns,
    "VIF": [
        variance_inflation_factor(vif_X.values, i)
        for i in range(vif_X.shape[1])
    ],
})
vif_table = vif_table[vif_table["Variable"] != "const"].reset_index(drop=True)
display(vif_table)
vif_table.to_csv(OUTPUT_DIR / "vif.csv", index=False)


## 6. Breusch–Pagan heteroskedasticity test

In [ ]:

bp_results = []

for outcome, (model, model_data, X) in models.items():
    lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(
        model.resid, model.model.exog
    )
    bp_results.append({
        "Outcome": outcome,
        "LM Statistic": lm_stat,
        "LM p-value": lm_pvalue,
        "F Statistic": f_stat,
        "F p-value": f_pvalue,
    })

bp_table = pd.DataFrame(bp_results)
display(bp_table)
bp_table.to_csv(OUTPUT_DIR / "breusch_pagan.csv", index=False)


## 7. Cook's distance influence check

Observations with Cook's distance greater than `4/n` are flagged as potentially influential, where `n` is the regression sample size.

In [ ]:

influence_rows = []

for outcome, (model, model_data, X) in models.items():
    influence = model.get_influence()
    cooks_d = influence.cooks_distance[0]
    n = len(model_data)
    threshold = 4 / n

    influence_rows.append({
        "Outcome": outcome,
        "N": n,
        "Threshold_4_over_n": threshold,
        "Max_Cooks_D": float(np.max(cooks_d)),
        "Influential_Count": int(np.sum(cooks_d > threshold)),
    })

influence_table = pd.DataFrame(influence_rows)
display(influence_table)
influence_table.to_csv(OUTPUT_DIR / "cook_influence.csv", index=False)


## 8. Sensitivity analysis

For ROE and Tobin's Q, the dependent variable is trimmed at the 1st and 99th percentiles and the regression is re-estimated using the same predictor specification and sector fixed effects.

In [ ]:

def fit_trimmed_model(outcome):
    cols = [outcome, *PREDICTORS, "Sector"]
    d = data[cols].dropna().copy()

    lower = d[outcome].quantile(0.01)
    upper = d[outcome].quantile(0.99)
    trimmed = d[d[outcome].between(lower, upper)].copy()
    trimmed["Sector"] = trimmed["Sector"].astype("category")

    X = pd.get_dummies(
        trimmed[PREDICTORS + ["Sector"]],
        columns=["Sector"],
        drop_first=True,
        dtype=float,
    )
    X = sm.add_constant(X, has_constant="add")
    y = trimmed[outcome].astype(float)

    return sm.OLS(y, X).fit(cov_type="HC3"), lower, upper

sensitivity_rows = []
for outcome in ["ROE", "Tobins_Q"]:
    trimmed_model, lower, upper = fit_trimmed_model(outcome)
    sensitivity_rows.append({
        "Outcome": outcome,
        "Original_N": models[outcome][0].nobs,
        "Trimmed_N": trimmed_model.nobs,
        "Lower_1pct": lower,
        "Upper_99pct": upper,
        "R_squared": trimmed_model.rsquared,
        "Adjusted_R_squared": trimmed_model.rsquared_adj,
    })

sensitivity_table = pd.DataFrame(sensitivity_rows)
display(sensitivity_table)
sensitivity_table.to_csv(OUTPUT_DIR / "sensitivity.csv", index=False)


## 9. Save a compact results workbook

This workbook provides the key statistical outputs generated by the notebook. The full coefficient tables for each model are also exported as CSV files.

In [ ]:

with pd.ExcelWriter(OUTPUT_DIR / "BUSI1783_analysis_results.xlsx", engine="openpyxl") as writer:
    descriptives.to_excel(writer, sheet_name="Descriptives")
    missingness.to_excel(writer, sheet_name="Missingness")
    correlations.to_excel(writer, sheet_name="Correlations")
    vif_table.to_excel(writer, sheet_name="VIF", index=False)
    bp_table.to_excel(writer, sheet_name="Breusch-Pagan", index=False)
    influence_table.to_excel(writer, sheet_name="Influence", index=False)
    sensitivity_table.to_excel(writer, sheet_name="Sensitivity", index=False)

    for outcome, (model, _, _) in models.items():
        coef = pd.DataFrame({
            "Coefficient": model.params,
            "Robust_SE_HC3": model.bse,
            "t": model.tvalues,
            "P_value": model.pvalues,
            "CI_Lower": model.conf_int()[0],
            "CI_Upper": model.conf_int()[1],
        })
        coef.to_excel(writer, sheet_name=f"{outcome}_Coefficients")

print("Results saved to:", OUTPUT_DIR.resolve())
